In [ ]:
import numpy as np
import pandas as pd
import re
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import TweetTokenizer
nltk.download('stopwords')
nltk.download('punkt')
nltk.download('punkt_tab')
import torch
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, RandomizedSearchCV, cross_val_score
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import accuracy_score
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from torch import nn
from transformers import AutoTokenizer, AutoModel
from gensim.models.fasttext import load_facebook_vectors
import random
import os
from collections import Counter

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\ASUS\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
c:\Users\ASUS\anaconda3\envs\nlp\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
SEED = 42

random.seed(SEED)
os.environ['PYTHONHASHSEED'] = str(SEED)

torch.manual_seed(SEED)
torch.cuda.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

rng = np.random.default_rng(seed=SEED)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

In [ ]:
stop_words = set(stopwords.words('english'))
label_to_num = {
  'Dislike': 0,
  'Mixed': 1,
  'Like': 2
}

def clean(text):
  text = re.sub(r'http\S+', '', text)       # strip URLs
  text = re.sub(r'#(\w+)', r'\1', text)      # keep hashtag text, drop #
  text = text.strip()
  return text

In [ ]:
data = pd.read_csv('./data/scraped-data.csv')

In [ ]:
rf_train, rf_test = train_test_split(data, test_size=0.2, train_size=0.8, random_state=42, shuffle=True, stratify=data['sentiment'])
rf_train_text = [[word for word in nltk.word_tokenize(clean(text)) if word not in stop_words] for text in rf_train['text']]
rf_test_text = [[word for word in nltk.word_tokenize(clean(text)) if word not in stop_words] for text in rf_test['text']]
rf_train_sentiment = list(map(lambda sent: label_to_num[sent], rf_train['sentiment']))
rf_test_sentiment = list(map(lambda sent: label_to_num[sent], rf_test['sentiment']))

# TF-IDF with Random Forest


In [ ]:
tfidf_vectorizer = TfidfVectorizer()

In [ ]:
def tfidf_vectorize(data, tfidf_vectorizer: TfidfVectorizer):
  X_tfidf = tfidf_vectorizer.fit_transform(data)
  words = tfidf_vectorizer.get_feature_names_out()
  X_tfidf_df = pd.DataFrame(X_tfidf.toarray())
  X_tfidf_df.columns = words
  return X_tfidf_df

In [ ]:
rf_train_X = tfidf_vectorize(rf_train_text, tfidf_vectorizer)
rf_test_X = tfidf_vectorizer.transform(rf_test_text)

In [ ]:
rf_parameters_dict = {
  'n_estimators': list(range(100, 800, 20)),
  'max_depth': [None, 5, 10, 15, 20, 30],
  'min_samples_split': list(range(2, 20)),
  'min_samples_leaf': list(range(1, 10)),
  'max_features': ['sqrt', 'log2', 0.3, 0.5, 0.7],
  'class_weight': ['balanced', 'balanced_subsample', None],
  'bootstrap': [True, False],
}

rf_clf = RandomForestClassifier(random_state=42, n_jobs=-1)

rf_cv = RandomizedSearchCV(
  rf_clf,
  rf_parameters_dict,
  n_iter=100,
  scoring='accuracy',
  cv=5,
  n_jobs=-1,
  random_state=42,
  verbose=True
)
rf_cv.fit(rf_train_X, np.array(rf_train_sentiment))

In [ ]:
rf_train_pred_Y = rf_cv.predict(rf_train_X)
rf_test_pred_Y = rf_cv.predict(rf_test_X)

In [ ]:
rf_train_acc = accuracy_score(rf_train_sentiment, rf_train_pred_Y)
rf_test_acc = accuracy_score(rf_test_sentiment, rf_test_pred_Y)

In [ ]:
print(f'RF Train Accuracy: {rf_train_acc*100:.4f}%')
print(f'RF Test Accuracy: {rf_test_acc*100:.4f}%')

In [ ]:
rf_train_cross_val_score = cross_val_score(rf_cv.best_estimator_, rf_train_X, rf_train_sentiment)
rf_test_cross_val_score = cross_val_score(rf_cv.best_estimator_, rf_test_X, rf_test_sentiment)

In [ ]:
plt.plot(figsize=(20, 8))
plt.title('Cross Val Scores')

plt.subplot(1, 2, 1)
plt.title('RandomForest Train Cross Val Accuracy')
plt.bar(rf_train_cross_val_score)
plt.ylabel('Accuracy')

plt.subplot(1, 2, 2)
plt.title('RandomForest Test Cross Val Accuracy')
plt.bar(rf_test_cross_val_score)
plt.ylabel('Accuracy')

plt.show()

# Encoder Model + XGBoost


In [ ]:
xgboost_train, xgboost_test = train_test_split(data, test_size=0.2, train_size=0.8, random_state=42, shuffle=True, stratify=data['sentiment'])
xgboost_train_text = [clean(text) for text in xgboost_train['text']]
xgboost_test_text = [clean(text) for text in xgboost_test['text']]
xgboost_train_sentiment = list(map(lambda sent: label_to_num[sent], xgboost_train['sentiment']))
xgboost_test_sentiment = list(map(lambda sent: label_to_num[sent], xgboost_test['sentiment']))

In [ ]:
xgboost_bertse_tokenizer = AutoTokenizer.from_pretrained("sentence-transformers/all-MiniLM-L6-v2")
xgboost_bertse_model = AutoModel.from_pretrained("sentence-transformers/all-MiniLM-L6-v2")

In [ ]:
def bertse_embed(texts: list[str], batch_size=32):
  all_embeddings = []
  with torch.no_grad():
    for i in range(0, len(texts), batch_size):
      batch = texts[i:i+batch_size]
      encoded = xgboost_bertse_tokenizer(batch, padding=True, truncation=True, return_tensors='pt')
      embeddings = xgboost_bertse_model(**encoded)
      mask = encoded['attention_mask'].unsqueeze(-1).float()
      summed = (embeddings.last_hidden_state * mask).sum()
      counts = mask.sum(1).clamp(min=1e-9)
      mean_pooled = summed / counts
      all_embeddings.append(mean_pooled)
    
  return torch.cat(all_embeddings).numpy()

In [ ]:
xgboost_train_X = bertse_embed(xgboost_train_text)
xgboost_test_X = bertse_embed(xgboost_test_text)

In [ ]:
xgboost_clf = XGBClassifier()

xgboost_parameters_dict = {
  'n_estimators': list(range(100, 600, 20)),
  'max_depth': list(range(3, 10)),
  'learning_rate': np.linspace(0.01, 0.3, 30).tolist(),
  'subsample': np.linspace(0.6, 1.0, 20).tolist(),
  'colsample_bytree': np.linspace(0.5, 1.0, 20).tolist(),
  'min_child_weight': list(range(1, 10)),
  'gamma': np.linspace(0, 0.5, 20).tolist(),
  'reg_alpha': np.linspace(0, 1, 20).tolist(),
  'reg_lambda': np.linspace(0, 2, 20).tolist(),
}

xgboost_cv = RandomizedSearchCV(
  xgboost_clf,
  xgboost_parameters_dict,
  n_iter=100,
  scoring='accuracy',
  cv=5,
  n_jobs=-1,
  random_state=42,
  verbose=True
)
xgboost_cv.fit(xgboost_train_X, np.array(xgboost_train_sentiment))

In [ ]:
xgboost_train_pred_Y = xgboost_cv.predict(xgboost_train_X)
xgboost_test_pred_Y = xgboost_cv.predict(xgboost_test_X)

In [ ]:
xgboost_train_acc = accuracy_score(xgboost_train_sentiment, xgboost_train_pred_Y)
xgboost_test_acc = accuracy_score(xgboost_test_sentiment, xgboost_test_pred_Y)

In [ ]:
print(f'XGBoost Train Accuracy: {xgboost_train_acc*100:.4f}%')
print(f'XGBoost Test Accuracy: {xgboost_test_acc*100:.4f}%')

In [ ]:
xgboost_train_cross_val_score = cross_val_score(xgboost_cv.best_estimator_, xgboost_train_X, xgboost_train_sentiment)
xgboost_test_cross_val_score = cross_val_score(xgboost_cv.best_estimator_, xgboost_test_X, xgboost_test_sentiment)

In [ ]:
plt.figure(figsize=(20, 8))
plt.title('Cross Val Scores')

plt.subplot(1, 2, 1)
plt.title('XGBoost Train Cross Val Accuracy')
plt.bar(range(len(xgboost_train_cross_val_score)), xgboost_train_cross_val_score)
plt.ylabel('Accuracy')

plt.subplot(1, 2, 2)
plt.title('XGBoost Test Cross Val Accuracy')
plt.bar(range(len(xgboost_test_cross_val_score)), xgboost_test_cross_val_score)
plt.ylabel('Accuracy')

plt.show()

# FastText + BiDirectional GRU


In [ ]:
gru_train, gru_temp = train_test_split(data, test_size=0.3, train_size=0.7, random_state=42, shuffle=True, stratify=data['sentiment'])
gru_val, gru_test = train_test_split(gru_temp, test_size=0.5, train_size=0.5, random_state=42, shuffle=True, stratify=data['sentiment'])
gru_train_text = [clean(text) for text in gru_train['text']]
gru_val_text = [clean(text) for text in gru_val['text']]
gru_test_text = [clean(text) for text in gru_test['text']]
gru_train_sentiment = list(map(lambda sent: label_to_num[sent], gru_train['sentiment']))
gru_val_sentiment = list(map(lambda sent: label_to_num[sent], gru_val['sentiment']))
gru_test_sentiment = list(map(lambda sent: label_to_num[sent], gru_test['sentiment']))

In [ ]:
wv = load_facebook_vectors('cc.en.300.bin')
tokenizer = TweetTokenizer(preserve_case=False, reduce_len=True, strip_handles=True)

gru_train_tokens = [tokenizer.tokenize(text) for text in gru_train_text]
gru_val_tokens = [tokenizer.tokenize(text) for text in gru_val_text]
gru_test_tokens = [tokenizer.tokenize(text) for text in gru_test_text]

word_counts = Counter(word for tokens in gru_train_tokens for word in tokens)
word_index = {'<PAD>': 0, '<UNK>': 1}
for word, _ in word_counts.most_common():
  word_index[word] = len(word_index)

embedding_dim = 300
vocab_size = len(word_index) + 1
embedding_matrix = np.zeros((vocab_size, embedding_dim))
for word, idx in word_index.items():
  if word in ['<PAD>', '<UNK>']:
    continue
  embedding_matrix[idx] = wv[word]

In [ ]:
def tokens_to_ids(tokens_list, word_index):
  return [[word_index.get(tok, word_index['<UNK>']) for tok in tokens] for tokens in tokens_list]

gru_train_ids = tokens_to_ids(gru_train_tokens, word_index)
gru_val_ids = tokens_to_ids(gru_val_tokens, word_index)
gru_test_ids = tokens_to_ids(gru_test_tokens, word_index)

gru_train_lengths = [min(len(seq), 50) for seq in gru_train_ids]
gru_val_lengths = [min(len(seq), 50) for seq in gru_val_ids]
gru_test_lengths = [min(len(seq), 50) for seq in gru_test_ids]

In [ ]:
class SentimentAnalysisDataset(nn.utils.data.Dataset):
  def __init__(self, sequences, labels, lengths, max_length=50, pad_idx=0):
    self.sequences = sequences
    self.labels = labels
    self.lengths = lengths
    self.max_length = max_length
    self.pad_idx = pad_idx
    
  def __len__(self):
    return len(self.sequences)
  
  def __getitem__(self, idx):
    sequence = self.sequences[idx]  # list of integer token ids
    label = self.labels[idx]
    length = min(self.lengths[idx], self.max_length)

    if len(sequence) < self.max_length:
      sequence = sequence + [self.pad_idx] * (self.max_length - len(sequence))
    else:
      sequence = sequence[:self.max_length]

    return (
      torch.tensor(sequence, dtype=torch.long),   # integer ids, model embeds these
      torch.tensor(label, dtype=torch.long),
      length,
    )

In [ ]:
train_dataset = SentimentAnalysisDataset(sequences=gru_train_ids, labels=gru_train_sentiment, lengths=gru_train_lengths)
train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=64, shuffle=True)
val_dataset = SentimentAnalysisDataset(sequences=gru_val_ids, labels=gru_val_sentiment, lengths=gru_val_lengths)
val_loader = torch.utils.data.DataLoader(val_dataset, batch_size=32, shuffle=False)
test_dataset = SentimentAnalysisDataset(sequences=gru_test_ids, labels=gru_test_sentiment, lengths=gru_test_lengths)
test_loader = torch.utils.data.DataLoader(test_dataset, batch_size=32, shuffle=False)

In [ ]:
class SentimentAnalysisBidirectionalGRU(nn.Module):
  def __init__(
    self,
    gru_hidden_size: int = 128,
    gru_num_layers: int = 1,
    gru_bidirectional: bool = True,
    gru_batch_first: bool = True,
    embedding_matrix: np.array = embedding_matrix,
    *args,  
    **kwargs
  ):
    super(SentimentAnalysisBidirectionalGRU, self).__init__(*args, **kwargs)
    
    vocab_size, embed_dim = embedding_matrix.shape
    self.embedding_matrix = embedding_matrix
    self.embed_dim = embed_dim
    self.vocab_size = vocab_size
    self.input_size = self.embed_dim
    self.gru_hidden_size = gru_hidden_size
    self.gru_num_layers = gru_num_layers
    self.gru_bidirectional = gru_bidirectional
    self.gru_batch_first = gru_batch_first
    
    self.EMBEDDING = nn.Embedding.from_pretrained(
      torch.tensor(self.embedding_matrix, dtype=torch.float32),
      freeze=True,
      padding_idx=0
    )
    self.GRU_UNIT = nn.GRU(
      input_size=self.input_size,
      hidden_size=self.gru_hidden_size,
      num_layers=self.gru_num_layers,
      bidirectional=self.gru_bidirectional,
      batch_first=self.gru_batch_first,
    )
    self.DROPOUT = nn.Dropout(0.3)
    self.GELU = nn.GELU()
    self.OUTPUT = nn.Linear(gru_hidden_size * 2, 3)
    self.SOFTMAX = torch.nn.Softmax(dim=1)
  
  def forward(self, X, lengths):
    embedded = self.EMBEDDING(X, enforce_sorted=False)
    packed = nn.utils.rnn.pack_padded_sequence(embedded, lengths, batch_first=True)
    _, hidden = self.GRU_UNIT(packed)
    hidden_cat = torch.cat([hidden[0], hidden[1]], dim=1)
    out = self.DROPOUT(self.GELU(hidden_cat))
    out = self.OUTPUT(out)
    
    return self.SOFTMAX(out)

In [ ]:
gru_clf = SentimentAnalysisBidirectionalGRU()

LEARNING_RATE = 2e-4
EPOCHS = 25
WEIGHT_DECAY = 1e-2

optimizer = torch.optim.AdamW(gru_clf.parameters(), LEARNING_RATE, weight_decay=WEIGHT_DECAY)
ce_loss = torch.nn.CrossEntropyLoss()

train_losses = []
val_losses = []

In [ ]:
for epoch in range(EPOCHS):
  epoch_train_loss = 0
  epoch_val_loss = 0
  
  for sequences, labels, lengths in train_loader:
    optimizer.zero_grad()
    gru_clf.train()
    
    pred_sentiment = gru_clf(sequences, lengths)
    loss = ce_loss(pred_sentiment, labels)
    loss.backward()
    epoch_train_loss += loss.item()
    
    optimizer.step()
    
  gru_clf.eval()
    
  with torch.no_grad():
    for val_sequences, val_labels, val_lengths in val_loader:
      val_pred_sentiment = gru_clf(val_sequences, val_lengths)
      val_loss = ce_loss(val_pred_sentiment, val_labels)
      epoch_val_loss += val_loss.item()
  
  train_losses.append(epoch_train_loss / len(train_loader))
  val_losses.append(epoch_val_loss / len(val_loader))

In [ ]:
plt.figure(figsize=(20, 8))
plt.plot(train_losses, label='Train Loss')
plt.plot(val_losses, label='Validation Loss')

plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.legend()

plt.show()

In [ ]:
train_batch_length = len(train_loader)
test_batch_length = len(test_loader)
train_acc = 0
test_acc = 0

gru_clf.eval()
with torch.no_grad():
  for sequences, labels, lengths in train_loader:
    pred_sentiment = gru_clf(sequences, lengths)
    pred_np = pred_sentiment.argmax(axis=-1).numpy()
    target_np = labels.numpy()
    
    train_acc += accuracy_score(target_np, pred_np)

  for test_sequences, test_labels, test_lengths in test_loader:
    test_pred_sentiment = gru_clf(test_sequences, test_lengths)
    test_pred_np = test_pred_sentiment.argmax(axis=-1).numpy()
    test_target_np = test_labels.numpy()
    
    test_acc += accuracy_score(test_target_np, test_pred_np)
  
  train_acc /= train_batch_length
  test_acc /= test_batch_length
  
print(f'Train Accuracy: {train_acc*100:.4f}%')
print(f'Test Accuracy: {test_acc*100:.4f}%')